## Torch dependencies

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

C:\Users\Daniel\Documents\Personal\breast_cancer_detection\.venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## MLFlow dependencies and initialization

In [3]:
import mlflow
from pprint import pprint

In [4]:
# Set here the URI from your MLFLow Tracking Server
TRACKING_URI = "http://localhost:5000"
client = mlflow.MlflowClient(tracking_uri=TRACKING_URI)
mlflow.set_tracking_uri(TRACKING_URI)

### Experiment creation

In [5]:
experiment_description = (
    "Training ResNet50 CNN for breat cancer detection."
    "This approach uses MLFlow instead of the custom pipeline built before."
    "This project has hyperparameters tunning using Optuna"
    "This experiment is using PNG images"
)

experiment_tags={
    "project_name": "breat-cancer-dection",
    "model_name": "resnet50",
    "mlflow.note.content": experiment_description,
    "parameters_tunner": "Optuna"
}

experiment_name = "ResNet_BreastCancerDection"

exp = client.get_experiment_by_name(experiment_name)

if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name, tags=experiment_tags
    )
else:
    exp_id = exp.experiment_id
    print(f"Experiment {experiment_name} already exists. Skipping creation")

Experiment ResNet_BreastCancerDection already exists. Skipping creation


## Training Dependencies

In [6]:
from pathlib import Path

# Force add the project root to sys.path (adjust as needed)
project_root = Path("../").resolve()  # one level up from /notebooks/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from tqdm import tqdm
from optuna.integration.mlflow import MLflowCallback


from datasets.cbisddsm import CBISDDSMDataset
from training.early_stopping import EarlyStopping
from training.engine import train_epoch, evaluate_epoch
from training.focal_loss import FocalLoss
from utils.to_tensor_16b import ToFloatTensor16Bit

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import optuna

import os
import time
import uuid

In [7]:
# Path to the dataset file
DATA_ABS_PATH     = os.path.abspath("D:/tfm/data")
IMAGES_ABS_PATH   = os.path.abspath("D:/tfm/data/CBIS-DDSM")
PNG_ABS_PATH      = os.path.abspath("D:/tfm/data/CBIS-DDSM-PNG")
CBISDDSM_FIXED_SET = DATA_ABS_PATH + '/meta/CBIS-DDSM-fixed.parquet'

### Set hyperparameters

In [8]:
num_epochs          = 1
train_batch_size    = 64
test_batch_size     = train_batch_size*2
val_batch_size      = train_batch_size*2
prefetch_factor     = 2
num_workers         = 8

learning_rate_l4    = 5e-5
learning_rate_fc    = 6.408966256267519e-05
scheduler_patience  = 10
alpha               = [1.0, 2.8619563143873994, 1.0]
early_stop_patience = 16
gamma               = 1.1120913750166153
dropout_rate        = 0.696670579337237
weight_decay        = 8.683454550720445e-05

early_stop_metric   = "val_recall"
early_stop_delta    = 0.001
early_stop_mode     = "max"

resize              = 224
horizontal_flip     = 0.5
degrees             = 10
# brightness          = 0.2
# contrast            = 0.2
kernel_size         = 3
normalize_mean      = [0.5, 0.5, 0.5]
normalize_std       = [0.5, 0.5, 0.5]

multi_view          = False
correlation_id      = uuid.uuid4()

In [9]:
transform = transforms.Compose([
    transforms.Resize((resize, resize)),
    transforms.RandomHorizontalFlip(horizontal_flip),
    transforms.RandomRotation(degrees=degrees),
    # transforms.ColorJitter(brightness=brightness, contrast=contrast),
    ToFloatTensor16Bit(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

eval_transform = transforms.Compose([
    transforms.Resize((resize, resize)),
    ToFloatTensor16Bit(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

### Set DataLoaders

In [10]:
# Dataframes
train_df = pd.read_parquet("train_png.parquet")
val_df = pd.read_parquet("val_png.parquet")
test_df = pd.read_parquet("test_png.parquet")

# PyTorch datasets
train_dataset = CBISDDSMDataset("train_png.parquet", transform=transform, multi_view=multi_view, images_base_path=PNG_ABS_PATH)
val_dataset   = CBISDDSMDataset("val_png.parquet", transform=eval_transform, multi_view=multi_view, images_base_path=PNG_ABS_PATH)
test_dataset  = CBISDDSMDataset("test_png.parquet", transform=eval_transform, multi_view=multi_view, images_base_path=PNG_ABS_PATH)

# PyTorch dataloaders
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset, batch_size=val_batch_size, shuffle=False, num_workers=num_workers, pin_memory=True,  persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, num_workers=num_workers, pin_memory=True,  persistent_workers=True)

# MLFlow datasets
ml_train_dataset = mlflow.data.from_pandas(train_df, name="cbis_ddsm_train")
ml_val_dataset = mlflow.data.from_pandas(val_df, name="cbis_ddsm_val")
ml_test_dataset = mlflow.data.from_pandas(test_df, name="cbis_ddsm_test")

## Using my stuff for training

In [11]:
import uuid
import optuna
import mlflow
import torch
import torchvision
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from optuna.integration.mlflow import MLflowCallback

mlflow.set_experiment(experiment_name)

NUM_STUDIES = 1
N_TRIALS_PER_STUDY = 1

# Track the global best across ALL studies and trials
global_best_malignant_recall = 0.0
global_best_run_id = None
corr_id = str(correlation_id)[:6]

# Single registered model name for all versions
REGISTERED_MODEL_NAME = "resnet50-breast-cancer" 

for study_number in range(NUM_STUDIES):
    study_name = f"resnet50_{study_number}_{str(correlation_id)[:6]}"
    
    with mlflow.start_run(run_name=f"parent_{study_name}", nested=False) as parent_run:
        parent_run_id = parent_run.info.run_id
        
        mlflow.set_tag("study_name", study_name)
        mlflow.log_param("study_number", study_number)

        study = optuna.create_study(
            study_name=study_name,
            direction="maximize",
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
            sampler=optuna.samplers.TPESampler(seed=42)
        )
        
        def objective(trial):
            global global_best_malignant_recall, global_best_run_id

            dropout_rate = trial.suggest_float("dropout_rate", 0.3, 0.8)
            learning_rate_fc = trial.suggest_float("learning_rate_fc", 1e-5, 1e-3, log=True)
            learning_rate_l4 = trial.suggest_float("learning_rate_l4", 1e-6, 1e-4, log=True)
            weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
            gamma = trial.suggest_float("gamma", 1.0, 2.5)
            alpha_bwc = trial.suggest_float("alpha_bwc", 1.0, 3.0)
            batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
            early_stop_patience = trial.suggest_int("early_stop_patience", 5, 20)
            correlation_id = str(uuid.uuid4())

            # Create nested run for this trial
            with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True) as trial_run:
                trial_run_id = trial_run.info.run_id
                
                mlflow.set_tag("study_name", study_name)
                mlflow.set_tag("parent_run_id", parent_run_id)
                mlflow.set_tag("trial_number", trial.number)
                
                # Log all parameters
                mlflow.log_params({
                    "trial_number": trial.number,
                    "correlation_id": correlation_id,
                    "dropout_rate": dropout_rate,
                    "learning_rate_fc": learning_rate_fc,
                    "learning_rate_l4": learning_rate_l4,
                    "weight_decay": weight_decay,
                    "gamma": gamma,
                    "alpha_bwc": alpha_bwc,
                    "alpha": str(alpha),
                    "batch_size": batch_size,
                    "early_stop_patience": early_stop_patience,
                    "num_epochs": num_epochs,
                    "num_workers": num_workers,
                    "prefetch_factor": prefetch_factor,
                    "scheduler_patience": scheduler_patience,
                    "resize": resize,
                    "horizontal_flip": horizontal_flip,
                    "degrees": degrees,
                    "normalize_mean": str(normalize_mean),
                    "normalize_std": str(normalize_std),
                    "multi_view": multi_view,
                    "early_stop_metric": early_stop_metric,
                    "early_stop_delta": early_stop_delta,
                    "early_stop_mode": early_stop_mode,
                })

                # Set up dataset/loaders
                transform = torchvision.transforms.Compose([
                    torchvision.transforms.Resize((resize, resize)),
                    torchvision.transforms.RandomHorizontalFlip(horizontal_flip),
                    torchvision.transforms.RandomRotation(degrees=degrees),
                    ToFloatTensor16Bit(),
                    torchvision.transforms.Normalize(mean=normalize_mean, std=normalize_std)
                ])
                eval_transform = torchvision.transforms.Compose([
                    torchvision.transforms.Resize((resize, resize)),
                    ToFloatTensor16Bit(),
                    torchvision.transforms.Normalize(mean=normalize_mean, std=normalize_std)
                ])

                train_dataset = CBISDDSMDataset(
                    "train_png.parquet",
                    transform=transform,
                    images_base_path=PNG_ABS_PATH,
                    multi_view=multi_view
                )
                val_dataset = CBISDDSMDataset(
                    "val_png.parquet",
                    transform=eval_transform,
                    images_base_path=PNG_ABS_PATH,
                    multi_view=multi_view
                )

                train_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    batch_size=batch_size,
                    shuffle=True,
                    num_workers=num_workers,
                    prefetch_factor=prefetch_factor
                )
                val_loader = torch.utils.data.DataLoader(
                    val_dataset,
                    batch_size=batch_size * 2,
                    shuffle=False,
                    num_workers=num_workers,
                    prefetch_factor=prefetch_factor
                )

                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                model = torchvision.models.resnet50(
                    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1
                )
                for param in model.parameters():
                    param.requires_grad = False
                
                model.fc = torch.nn.Sequential(
                    torch.nn.Dropout(dropout_rate),
                    torch.nn.Linear(model.fc.in_features, 3)
                )
                
                for layer in [model.layer4, model.fc]:
                    for param in layer.parameters():
                        param.requires_grad = True
                
                model = model.to(device)
                
                optimizer = Adam([
                    {'params': model.layer4.parameters(), 'lr': learning_rate_l4},
                    {'params': model.fc.parameters(), 'lr': learning_rate_fc}
                ], weight_decay=weight_decay)
                
                scheduler = ReduceLROnPlateau(
                    optimizer, mode="max", factor=0.5, 
                    patience=scheduler_patience, min_lr=1e-6
                )
                criterion = FocalLoss(gamma=gamma, alpha=alpha)
                early_stopping = EarlyStopping(
                    monitor=early_stop_metric, mode=early_stop_mode, 
                    patience=early_stop_patience, delta=early_stop_delta
                )

                best_malignant_recall = 0
                best_model_wts = None

                for epoch in range(num_epochs):
                    print(f"\nStudy {study_number} - Trial {trial.number} - Epoch {epoch+1}/{num_epochs}")
                    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
                    val_acc, val_loss, val_recall, val_precision, val_f1, val_auc, _, _, val_class_metrics = evaluate_epoch(
                        model, val_loader, criterion, device
                    )

                    val_malignant_recall = val_class_metrics.get('MALIGNANT', {}).get('recall', 0) or \
                                          val_class_metrics.get(2, {}).get('recall', 0) or \
                                          val_class_metrics.get('class_2', {}).get('recall', 0)

                    print(f"This is val_malignant_recall: {val_malignant_recall}")

                    if val_malignant_recall > best_malignant_recall:
                        best_malignant_recall = val_malignant_recall
                        best_model_wts = model.state_dict()

                    # Log metrics
                    mlflow.log_metric("train_loss", train_loss, step=epoch)
                    mlflow.log_metric("val_loss", val_loss, step=epoch)
                    mlflow.log_metric("val_accuracy", val_acc, step=epoch)
                    mlflow.log_metric("val_recall", val_recall, step=epoch)
                    mlflow.log_metric("val_precision", val_precision, step=epoch)
                    mlflow.log_metric("val_f1", val_f1, step=epoch)
                    mlflow.log_metric("val_auc", val_auc, step=epoch)
                    mlflow.log_metric("val_malignant_recall", val_malignant_recall, step=epoch)

                    # Log per-class metrics
                    for class_name, metrics in val_class_metrics.items():
                        for metric_name, metric_value in metrics.items():
                            mlflow.log_metric(f"val_{class_name}_{metric_name}", metric_value, step=epoch)

                    scheduler.step(val_malignant_recall)

                    # Report to Optuna for pruning (using malignant recall)
                    trial.report(val_malignant_recall, epoch)
                    if trial.should_prune():
                        mlflow.log_param("pruned", True)
                        mlflow.log_param("pruned_at_epoch", epoch)
                        raise optuna.TrialPruned()

                    # Early stopping based on malignant recall
                    if early_stopping.step(val_malignant_recall):
                        print(f"Early stopping at epoch {epoch+1}")
                        mlflow.log_param("early_stopped", True)
                        mlflow.log_param("early_stopped_at_epoch", epoch)
                        break

                if best_model_wts is not None:
                    model.load_state_dict(best_model_wts)

                val_acc, val_loss, val_recall, val_precision, val_f1, val_auc, val_views, val_view_predictions, val_class_metrics = evaluate_epoch(
                    model, val_loader, criterion, device
                )

                final_malignant_recall = val_class_metrics.get('MALIGNANT', {}).get('recall', 0) or \
                                        val_class_metrics.get(2, {}).get('recall', 0) or \
                                        val_class_metrics.get('class_2', {}).get('recall', 0)

                # Log final metrics
                mlflow.log_metric("final_val_loss", val_loss)
                mlflow.log_metric("final_val_accuracy", val_acc)
                mlflow.log_metric("final_val_recall", val_recall)
                mlflow.log_metric("final_val_precision", val_precision)
                mlflow.log_metric("final_val_f1", val_f1)
                mlflow.log_metric("final_val_auc", val_auc)
                mlflow.log_metric("final_val_malignant_recall", final_malignant_recall)

                for class_name, metrics in val_class_metrics.items():
                    for metric_name, metric_value in metrics.items():
                        mlflow.log_metric(f"final_val_{class_name}_{metric_name}", metric_value)

                # Always log the PyTorch model artifact (for tracking)
                mlflow.pytorch.log_model(
                    pytorch_model=model,
                    artifact_path="pytorch_model"
                )

                # Only REGISTER with image predictor wrapper if it beats the global best
                if final_malignant_recall > global_best_malignant_recall:
                    print(f"\nNEW BEST! Malignant Recall: {final_malignant_recall:.4f}")
                    global_best_malignant_recall = final_malignant_recall
                    global_best_run_id = trial_run_id
                    
                    from inference.resnet_image_predictor import ResNetImagePredictor
                    
                    # Save model to temp location, then log wrapper
                    import tempfile
                    import os
                    
                    with tempfile.TemporaryDirectory() as tmpdir:
                        temp_model_path = os.path.join(tmpdir, "temp_pytorch_model")
                        mlflow.pytorch.save_model(model, temp_model_path)
                        
                        mlflow.pyfunc.log_model(
                            artifact_path="image_predictor",
                            python_model=ResNetImagePredictor(),
                            artifacts={"pytorch_model": temp_model_path},
                            pip_requirements=[
                                f'mlflow=={mlflow.__version__}',
                                f'torch=={torch.__version__}',
                                f'torchvision=={torchvision.__version__}',
                                'pillow', 'pydicom', 'numpy', 'pandas', 'opencv-python'
                            ]
                        )
                    
                    model_uri = f"runs:/{trial_run_id}/image_predictor"
                    mlflow.register_model(model_uri=model_uri, name=REGISTERED_MODEL_NAME)
                    mlflow.set_tag("is_best_model", True)
                    mlflow.log_param("global_best_malignant_recall", final_malignant_recall)
                else:
                    # For non-best trials, just log PyTorch model for tracking
                    mlflow.pytorch.log_model(
                        pytorch_model=model,
                        artifact_path="pytorch_model"
                    )
                    mlflow.set_tag("is_best_model", False)

                print(f"Trial {trial.number} complete - Malignant Recall: {final_malignant_recall:.4f}, Val AUC: {val_auc:.4f}")
                
                # Return malignant recall for Optuna optimization
                return final_malignant_recall

        # Run the optimization for this study
        study.optimize(objective, n_trials=N_TRIALS_PER_STUDY, show_progress_bar=True)

        # After the study finishes, log summary under the parent run
        mlflow.log_param("best_trial_number", study.best_trial.number)
        mlflow.log_param("best_malignant_recall", study.best_value)
        mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
        
        print(f"\nStudy {study_number} complete. Best malignant recall: {study.best_value:.4f}")

print(f"\n{'='*80}")
print(f"All studies completed.")
print(f"Global best malignant recall: {global_best_malignant_recall:.4f}")
print(f"Global best run ID: {global_best_run_id}")
print(f"Registered model: {REGISTERED_MODEL_NAME}")
print(f"{'='*80}")

[I 2025-11-16 18:31:37,677] A new study created in memory with name: resnet50_0_e0c504


  0%|          | 0/1 [00:00<?, ?it/s]


Study 0 - Trial 0 - Epoch 1/1


Training:   0%|          | 0/37 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

...

2025/11/16 18:33:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 18:33:41 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu129) contains a local version label (+cu129). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/11/16 18:33:45 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.23.0+cu129) contains a local version label (+cu129). MLflow logged a pip requirement for this package as 'torchvision==0.23.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/11/16 18:33:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example`


NEW BEST! Malignant Recall: 0.4126


2025/11/16 18:33:52 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.23.0+cu129) contains a local version label (+cu129). MLflow logged a pip requirement for this package as 'torchvision==0.23.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/11/16 18:33:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2025/11/16 18:33:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'resnet50-breast-cancer' already exists. Creating a new version of this model...
2025/11/16 18:33:59 WARNING mlflow.tracking._model_registry.fluent: Run with id fcaaec4a8e9641caa8d7fc08c3fc0556 has no artifacts at artifact path 'image_predictor', registering model based on models:/m-430d3aafaafb4863a3d65c07645ff66e instead
2025/11/16 18:33:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: resnet50-breast-cancer, version 2
Created version '2' of model 'resnet50-breast-cancer'.


Trial 0 complete - Malignant Recall: 0.4126, Val AUC: 0.6379
🏃 View run trial_0 at: http://localhost:5000/#/experiments/5/runs/fcaaec4a8e9641caa8d7fc08c3fc0556
🧪 View experiment at: http://localhost:5000/#/experiments/5
[I 2025-11-16 18:33:59,814] Trial 0 finished with value: 0.4125560538116592 and parameters: {'dropout_rate': 0.48727005942368123, 'learning_rate_fc': 0.0007969454818643932, 'learning_rate_l4': 2.9106359131330718e-05, 'weight_decay': 0.00015751320499779721, 'gamma': 1.2340279606636548, 'alpha_bwc': 1.3119890406724053, 'batch_size': 64, 'early_stop_patience': 5}. Best is trial 0 with value: 0.4125560538116592.

Study 0 complete. Best malignant recall: 0.4126
🏃 View run parent_resnet50_0_e0c504 at: http://localhost:5000/#/experiments/5/runs/258d6b4027f644afa1e82bc4763a5ee1
🧪 View experiment at: http://localhost:5000/#/experiments/5

All studies completed.
Global best malignant recall: 0.4126
Global best run ID: fcaaec4a8e9641caa8d7fc08c3fc0556
Registered model: resnet50-br

In [12]:
import mlflow
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# ===== LOAD BEST MODEL =====
print("\n" + "="*60)
print("LOADING BEST MODEL FOR TEST EVALUATION")
print("="*60)
print(f"\nGlobal best malignant recall: {global_best_malignant_recall:.4f}")
print(f"Global best run ID: {global_best_run_id}")

# Load the best model from MLflow
model_uri = f"runs:/{global_best_run_id}/pytorch_model"
model = mlflow.pytorch.load_model(model_uri)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Get the best run's parameters
client = mlflow.tracking.MlflowClient()
run = client.get_run(global_best_run_id)
best_params = run.data.params

print("\nBest hyperparameters used:")
for key in ['dropout_rate', 'learning_rate_fc', 'learning_rate_l4', 
            'weight_decay', 'gamma', 'alpha_bwc', 'batch_size']:
    if key in best_params:
        print(f"  {key}: {best_params[key]}")

# ===== PREPARE TEST DATASET =====
print("\n" + "="*60)
print("PREPARING TEST DATASET")
print("="*60)

# Use the same transforms as validation (no augmentation)
resize = 224
normalize_mean = [0.5, 0.5, 0.5]
normalize_std = [0.5, 0.5, 0.5]

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((resize, resize)),
    ToFloatTensor16Bit(),
    torchvision.transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

# ===== EVALUATE ON TEST SET =====
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

# Recreate the loss function with the same parameters
gamma = float(best_params.get('gamma', 2.0))
alpha_bwc = float(best_params.get('alpha_bwc', 2.0))
alpha = [1.0, alpha_bwc, 1.0]
criterion = FocalLoss(gamma=gamma, alpha=alpha)

# Evaluate
test_acc, test_loss, test_recall, test_precision, test_f1, test_auc, test_views, test_view_predictions, test_class_metrics = evaluate_epoch(
    model, test_loader, criterion, device
)

# Extract malignant recall
test_malignant_recall = test_class_metrics.get('MALIGNANT', {}).get('recall', 0) or \
                        test_class_metrics.get(2, {}).get('recall', 0) or \
                        test_class_metrics.get('class_2', {}).get('recall', 0)

# ===== DISPLAY RESULTS =====
print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"\nOverall Metrics:")
print(f"  Accuracy:  {test_acc:.4f}")
print(f"  Loss:      {test_loss:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  F1 Score:  {test_f1:.4f}")
print(f"  AUC:       {test_auc:.4f}")

print(f"\nCRITICAL METRIC:")
print(f"  Malignant (Class 2) Recall: {test_malignant_recall:.4f}")

print("\nPer-Class Metrics:")
for class_name, metrics in test_class_metrics.items():
    print(f"\n  {class_name}:")
    for metric_name, value in metrics.items():
        print(f"    {metric_name}: {value:.4f}")

# ===== LOG TEST RESULTS TO MLFLOW =====
print("\n" + "="*60)
print("LOGGING TEST RESULTS TO MLFLOW")
print("="*60)

# Create a new run to log test results
mlflow.set_experiment(experiment_name)
with mlflow.start_run(run_name=f"test_evaluation_{corr_id}") as test_run:
    mlflow.set_tag("evaluation_type", "test_set")
    mlflow.set_tag("best_model_run_id", global_best_run_id)
    
    # Log test metrics
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_recall", test_recall)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("test_f1", test_f1)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("test_malignant_recall", test_malignant_recall)
    
    # Log per-class metrics
    for class_name, metrics in test_class_metrics.items():
        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(f"test_{class_name}_{metric_name}", metric_value)
    
    # ===== CREATE VISUALIZATIONS =====
    
    # 1. Confusion Matrix (if you have predictions and labels)
    # You may need to modify evaluate_epoch to return all predictions and labels
    # For now, this is a placeholder structure
    
    # 2. Performance Comparison: Validation vs Test
    comparison_data = {
        'Metric': ['Accuracy', 'Recall', 'Precision', 'F1', 'AUC', 'Malignant Recall'],
        'Validation': [
            float(run.data.metrics.get('final_val_accuracy', 0)),
            float(run.data.metrics.get('final_val_recall', 0)),
            float(run.data.metrics.get('final_val_precision', 0)),
            float(run.data.metrics.get('final_val_f1', 0)),
            float(run.data.metrics.get('final_val_auc', 0)),
            float(run.data.metrics.get('final_val_malignant_recall', global_best_malignant_recall))
        ],
        'Test': [test_acc, test_recall, test_precision, test_f1, test_auc, test_malignant_recall]
    }
    
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(comparison_data['Metric']))
    width = 0.35
    
    ax.bar(x - width/2, comparison_data['Validation'], width, label='Validation', alpha=0.8)
    ax.bar(x + width/2, comparison_data['Test'], width, label='Test', alpha=0.8)
    
    ax.set_xlabel('Metrics', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Model Performance: Validation vs Test', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_data['Metric'], rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plt.savefig(f"test_vs_validation_performancee_{corr_id}.png", dpi=300, bbox_inches='tight')
    mlflow.log_artifact(f"test_vs_validation_performancee_{corr_id}.png")
    plt.close()
    
    # 3. Per-Class Performance
    class_names = list(test_class_metrics.keys())
    recalls = [test_class_metrics[c].get('recall', 0) for c in class_names]
    precisions = [test_class_metrics[c].get('precision', 0) for c in class_names]
    f1_scores = [test_class_metrics[c].get('f1', 0) for c in class_names]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(class_names))
    width = 0.25
    
    ax.bar(x - width, recalls, width, label='Recall', alpha=0.8)
    ax.bar(x, precisions, width, label='Precision', alpha=0.8)
    ax.bar(x + width, f1_scores, width, label='F1 Score', alpha=0.8)
    
    ax.set_xlabel('Class', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Per-Class Performance on Test Set', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1.1])
    plt.tight_layout()
    
    plt.savefig(f"test_per_class_performance_{corr_id}.png", dpi=300, bbox_inches='tight')
    mlflow.log_artifact(f"test_per_class_performance_{corr_id}.png")
    plt.close()
    
    print("Test results and visualizations logged to MLflow")

print("\n" + "="*60)
print("TEST EVALUATION COMPLETE")
print("="*60)
print(f"\nFinal Test Malignant Recall: {test_malignant_recall:.4f}")
print(f"Registered Model: {REGISTERED_MODEL_NAME}")
print(f"\nServe with:")
print(f'$env:MLFLOW_TRACKING_URI="http://localhost:5000"; mlflow models serve -m "models:/{REGISTERED_MODEL_NAME}/latest" --port 5003 --env-manager=local')
print("="*60)


LOADING BEST MODEL FOR TEST EVALUATION

Global best malignant recall: 0.4126
Global best run ID: fcaaec4a8e9641caa8d7fc08c3fc0556



Best hyperparameters used:
  dropout_rate: 0.48727005942368123
  learning_rate_fc: 0.0007969454818643932
  learning_rate_l4: 2.9106359131330718e-05
  weight_decay: 0.00015751320499779721
  gamma: 1.2340279606636548
  alpha_bwc: 1.3119890406724053
  batch_size: 64

PREPARING TEST DATASET

EVALUATING ON TEST SET


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


TEST SET RESULTS

Overall Metrics:
  Accuracy:  0.4510
  Loss:      0.6184
  Recall:    0.4475
  Precision: 0.4470
  F1 Score:  0.4410
  AUC:       0.6519

CRITICAL METRIC:
  Malignant (Class 2) Recall: 0.3144

Per-Class Metrics:

  class_0:
    recall: 0.5613
    precision: 0.4567
    f1: 0.5036
    accuracy: 0.5058
    auc_roc: 0.5173

  class_1:
    recall: 0.4667
    precision: 0.4308
    f1: 0.4480
    accuracy: 0.8012
    auc_roc: 0.7850

  class_2:
    recall: 0.3144
    precision: 0.4536
    f1: 0.3714
    accuracy: 0.5951
    auc_roc: 0.6534

LOGGING TEST RESULTS TO MLFLOW
Test results and visualizations logged to MLflow
🏃 View run test_evaluation_e0c504 at: http://localhost:5000/#/experiments/5/runs/5fe2deeb1343470bb4128453223eff24
🧪 View experiment at: http://localhost:5000/#/experiments/5

TEST EVALUATION COMPLETE

Final Test Malignant Recall: 0.3144
Registered Model: resnet50-breast-cancer

Serve with:
$env:MLFLOW_TRACKING_URI="http://localhost:5000"; mlflow models serve 